In [ ]:
# | default_exp eval_engine

In [ ]:
# | export
from __future__ import annotations
import pandas as pd
import numpy as np
from scipy import stats
import structlog
import traceback

# Visualizations
import plotly.express as px
import plotly.graph_objects as go

# Sklearn Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

log = structlog.get_logger()

In [ ]:
# | export
class FeatureEvaluator:
    """Base class for all feature evaluators.
    Defines the extraction contract that transforms raw DuckDB queries into 1D arrays.
    """

    name: str = "base"
    source_file: str = ".dummy.parquet"
    tier: int = 1
    category: str = "general"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        """Transform the loaded raw dataframe into meaningful scalar metrics.
        Called per sample-group or per sample."""
        raise NotImplementedError(
            "Feature evaluators must implement the extract() method."
        )


def parse_array(s) -> list[float]:
    """Parse a string-encoded numeric array into a list of floats.

    Handles formats like '[1.0 2.0 3.0]' from parquet serialization.
    Returns empty list on any parse failure (no silent corruption).
    """
    if not isinstance(s, str) or not s.startswith("["):
        return []
    clean = (
        s.replace("[", "").replace("]", "").replace(chr(10), "").replace(chr(13), "")
    )
    try:
        return [float(x) for x in clean.split()]
    except (ValueError, TypeError):
        return []


def univariate_auc(
    feature_col,
    y,
    n_folds: int = 5,
    random_state: int = 42,
) -> float:
    """Compute cross-validated AUC for a single feature using univariate LR.

    Args:
        feature_col: pandas Series or array-like of a single feature.
        y: binary label array (0/1).
        n_folds: number of CV folds.
        random_state: random seed.

    Returns:
        Cross-validated AUC (float). Returns 0.5 if the feature is constant,
        there are too few samples per class, or CV fails.
    """
    import numpy as np

    X = np.array(feature_col).reshape(-1, 1)
    # Replace NaN with 0 for safety
    X = np.nan_to_num(X, nan=0.0)
    y = np.array(y, dtype=int)

    if np.std(X) == 0:
        return 0.5

    try:
        min_class = np.bincount(y).min()
        folds = min(n_folds, min_class)
        if folds < 2:
            return 0.5
        pipe = Pipeline(
            [("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))]
        )
        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)
        proba = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
        return float(roc_auc_score(y, proba))
    except Exception:
        return 0.5

In [ ]:
# | export
# Only the 4 ML-relevant tiers. "Insufficient Data" samples are
# excluded from all statistical tests and ML models by design.
LABEL_ORDER = ["True ctDNA+", "Possible ctDNA+", "Possible ctDNA−", "Healthy Normal"]
NEON_COLORS = {
    "True ctDNA+": "#E85D75",  # Pinkish red
    "Possible ctDNA+": "#FF9B42",  # Vivid orange
    "Possible ctDNA−": "gray",
    "Possible ctDNA-": "gray",
    "Healthy Normal": "#8DDA77",  # Bright green
}

CVD_SAFE_COLORS = {
    "True ctDNA+": "#D55E00",  # Vermillion (CVD-safe)
    "Possible ctDNA+": "#E69F00",  # Orange (CVD-safe)
    "Possible ctDNA−": "#999999",  # Gray (CVD-safe)
    "Possible ctDNA-": "#999999",  # Gray fallback
    "Healthy Normal": "#009E73",  # Bluish Green (CVD-safe)
}

LABEL_COLORS = NEON_COLORS.copy()


def set_theme(cvd_safe: bool = False):
    """Dynamically updates the global label and model colors based on CVD preference."""
    global LABEL_COLORS
    LABEL_COLORS.update(CVD_SAFE_COLORS if cvd_safe else NEON_COLORS)

In [ ]:
# | export
def evaluate_feature(
    feature_values: pd.Series,
    labels: pd.Series,
    total_fragments: pd.Series | None = None,
    max_vaf: pd.Series | None = None,
) -> dict:
    """Run all statistical tests for a single feature in one stratum.
    Outputs metrics directly to scoring dict.
    """
    try:
        groups = {}
        for label in LABEL_ORDER:
            mask = labels == label
            # Ensure index alignment if possible
            vals = feature_values[mask].dropna()
            groups[label] = vals

        result = {}

        # --- Kruskal-Wallis (4-group) ---
        non_empty = [g for g in groups.values() if len(g) >= 2]
        if len(non_empty) >= 2:
            try:
                kw_stat, kw_p = stats.kruskal(*non_empty)
                result["kw_statistic"] = kw_stat
                result["kw_pvalue"] = kw_p
            except ValueError as e:
                log.warning("kruskal_failed", error=str(e))

        # --- Pairwise Mann-Whitney U ---
        pairs = [
            ("True ctDNA+", "Healthy Normal"),
            ("Possible ctDNA+", "Healthy Normal"),
            ("True ctDNA+", "Possible ctDNA+"),
            ("Possible ctDNA−", "Healthy Normal"),
            ("True ctDNA+", "Possible ctDNA−"),
        ]
        for g1_name, g2_name in pairs:
            g1, g2 = groups.get(g1_name, []), groups.get(g2_name, [])
            key = (
                f"mwu_{g1_name}_vs_{g2_name}".replace(" ", "_")
                .replace("+", "pos")
                .replace("−", "neg")
            )
            if len(g1) >= 2 and len(g2) >= 2:
                try:
                    u_stat, p_val = stats.mannwhitneyu(g1, g2, alternative="two-sided")
                    r = 1 - (2 * u_stat) / (
                        len(g1) * len(g2)
                    )  # rank-biserial. Note: g1 is always the biologically more "positive" tier (e.g. True+). Positive r means g2 tends to be larger, negative r means g1 tends to be larger.
                    result[f"{key}_pvalue"] = p_val
                    result[f"{key}_effect_size"] = r
                except ValueError as e:
                    log.debug("mwu_failed", pair=f"{g1_name}-{g2_name}", error=str(e))

        # --- FDR correction on pairwise p-values (Benjamini-Hochberg) ---
        pairwise_pval_keys = [
            k for k in result if k.endswith("_pvalue") and k.startswith("mwu_")
        ]
        if len(pairwise_pval_keys) >= 2:
            try:
                from statsmodels.stats.multitest import multipletests

                raw_pvals = [result[k] for k in pairwise_pval_keys]
                _, corrected, _, _ = multipletests(raw_pvals, method="fdr_bh")
                for k, p_corr in zip(pairwise_pval_keys, corrected):
                    result[k.replace("_pvalue", "_pvalue_fdr")] = float(p_corr)
            except ImportError:
                log.warning("statsmodels_not_installed", note="FDR correction skipped")
            except Exception as e:
                log.warning("fdr_correction_failed", error=str(e))

        # --- Cohen's d (True ctDNA+ vs Healthy) ---
        true_pos = groups.get("True ctDNA+", pd.Series())
        healthy = groups.get("Healthy Normal", pd.Series())
        if len(true_pos) >= 2 and len(healthy) >= 2:
            pooled_std = np.sqrt(
                (
                    (len(true_pos) - 1) * true_pos.std() ** 2
                    + (len(healthy) - 1) * healthy.std() ** 2
                )
                / (len(true_pos) + len(healthy) - 2)
            )
            if pooled_std > 0:
                result["cohens_d_true_vs_healthy"] = (
                    true_pos.mean() - healthy.mean()
                ) / pooled_std

        # --- Cohen's d (Possible ctDNA+ vs Healthy) ---
        poss_pos = groups.get("Possible ctDNA+", pd.Series())
        if len(poss_pos) >= 2 and len(healthy) >= 2:
            pooled_std_pp = np.sqrt(
                (
                    (len(poss_pos) - 1) * poss_pos.std() ** 2
                    + (len(healthy) - 1) * healthy.std() ** 2
                )
                / (len(poss_pos) + len(healthy) - 2)
            )
            if pooled_std_pp > 0:
                result["cohens_d_possible_vs_healthy"] = (
                    poss_pos.mean() - healthy.mean()
                ) / pooled_std_pp

        # --- Spearman correlations (Confounders) ---
        if max_vaf is not None:
            valid = feature_values.notna() & max_vaf.notna() & (max_vaf > 0)
            if valid.sum() >= 10:
                try:
                    if (
                        np.var(feature_values[valid]) == 0
                        or np.var(max_vaf[valid]) == 0
                    ):
                        r_val, p_val = 0.0, 1.0
                    else:
                        r_val, p_val = stats.spearmanr(
                            feature_values[valid], max_vaf[valid]
                        )
                    result["spearman_vaf_r"] = r_val
                    result["spearman_vaf_p"] = p_val
                except ValueError as e:
                    log.warning("spearman_vaf_failed", error=str(e))

        if total_fragments is not None:
            valid = feature_values.notna() & total_fragments.notna()
            if valid.sum() >= 10:
                try:
                    if (
                        np.var(feature_values[valid]) == 0
                        or np.var(total_fragments[valid]) == 0
                    ):
                        r_val, p_val = 0.0, 1.0
                    else:
                        r_val, p_val = stats.spearmanr(
                            feature_values[valid], total_fragments[valid]
                        )
                    result["spearman_depth_r"] = r_val
                    result["spearman_depth_p"] = p_val
                except ValueError as e:
                    log.warning("spearman_depth_failed", error=str(e))

        # --- Sample counts ---
        for label, vals in groups.items():
            key = label.replace(" ", "_").replace("+", "pos").replace("−", "neg")
            result[f"n_{key}"] = len(vals)

        result["low_power"] = any(len(v) < 10 for v in groups.values())

        # --- Data quality metrics ---
        result["n_missing"] = int(feature_values.isna().sum())
        result["pct_missing"] = float(feature_values.isna().mean() * 100)
        non_null = feature_values.dropna()
        result["is_zero_variance"] = bool(len(non_null) < 2 or non_null.std() == 0)

        return result
    except Exception as e:
        log.error("evaluate_feature_failed", error=str(e), trace=traceback.format_exc())
        return {"error": str(e)}

In [ ]:
# | export
def plot_violin(
    df: pd.DataFrame, feature_col: str, label_col: str = "label", title: str = ""
) -> go.Figure:
    """4-group violin with overlaid box plot and individual points for small groups."""
    try:
        fig = px.violin(
            df,
            x=label_col,
            y=feature_col,
            color=label_col,
            box=True,
            points="outliers",
            category_orders={label_col: LABEL_ORDER},
            color_discrete_map=LABEL_COLORS,
            title=title,
        )
        fig.update_layout(showlegend=False, xaxis_title="", yaxis_title=feature_col)
        return fig
    except Exception as e:
        log.error("plot_violin_failed", feature=feature_col, error=str(e))
        return go.Figure()


def plot_density(
    df: pd.DataFrame, feature_col: str, label_col: str = "label", title: str = ""
) -> go.Figure:
    """Overlaid density curves per group — shows distribution shape differences."""
    try:
        fig = go.Figure()
        for label in LABEL_ORDER:
            if label not in df[label_col].values:
                continue
            subset = df[df[label_col] == label][feature_col].dropna()
            if len(subset) > 5:
                fig.add_trace(
                    go.Violin(
                        y=subset,
                        name=label,
                        side="positive",
                        line_color=LABEL_COLORS[label],
                        meanline_visible=True,
                        scalemode="width",
                        width=0.8,
                    )
                )
        fig.update_layout(title=title, violinmode="overlay")
        return fig
    except Exception as e:
        log.error("plot_density_failed", feature=feature_col, error=str(e))
        return go.Figure()


def plot_feature_vs_vaf(
    df: pd.DataFrame,
    feature_col: str,
    vaf_col: str = "max_vaf",
    label_col: str = "label",
    title: str = "",
) -> go.Figure:
    """Continuous relationship between feature and tumor burden (VAF proxy)."""
    try:
        cancer = df[df[vaf_col] > 0]  # exclude healthy & zero-VAF
        if cancer.empty:
            return go.Figure()
        fig = px.scatter(
            cancer,
            x=vaf_col,
            y=feature_col,
            color=label_col,
            color_discrete_map=LABEL_COLORS,
            opacity=0.5,
            trendline="lowess",
            log_x=True,
            title=title,
        )
        return fig
    except Exception as e:
        log.error("plot_vs_vaf_failed", feature=feature_col, error=str(e))
        return go.Figure()


def plot_roc_curves(
    y_true_dict: dict[str, np.ndarray],
    y_score_dict: dict[str, np.ndarray],
    title: str = "",
) -> go.Figure:
    """Overlay ROC curves for multiple comparisons."""
    try:
        fig = go.Figure()
        for name, y_true in y_true_dict.items():
            if name not in y_score_dict:
                continue
            y_score = y_score_dict[name]
            if len(np.unique(y_true)) < 2:
                continue  # Cannot plot ROC with 1 class
            fpr, tpr, _ = roc_curve(y_true, y_score)
            roc_auc = auc(fpr, tpr)
            fig.add_trace(
                go.Scatter(
                    x=fpr,
                    y=tpr,
                    name=f"{name} (AUC={roc_auc:.3f})",
                    mode="lines",
                )
            )
        fig.add_trace(
            go.Scatter(
                x=[0, 1],
                y=[0, 1],
                mode="lines",
                line=dict(dash="dash", color="gray"),
                name="Random",
            )
        )
        fig.update_layout(
            title=title,
            xaxis_title="False Positive Rate",
            yaxis_title="True Positive Rate",
        )
        return fig
    except Exception as e:
        log.error("plot_roc_curves_failed", error=str(e))
        return go.Figure()


def plot_feature_importance(
    importances: dict[str, float], title: str = ""
) -> go.Figure:
    """Bar plot of RF feature importances."""
    try:
        df = pd.DataFrame(
            {
                "feature": list(importances.keys()),
                "importance": list(importances.values()),
            }
        )
        df = df.sort_values("importance", ascending=True).tail(20)  # Top 20
        fig = px.bar(df, x="importance", y="feature", orientation="h", title=title)
        return fig
    except Exception as e:
        log.error("plot_rf_importance_failed", error=str(e))
        return go.Figure()


def plot_threshold_sensitivity(results_df: pd.DataFrame, title: str = "") -> go.Figure:
    """Show how label counts shift with VAF/min_variants thresholds."""
    try:
        fig = px.line(
            results_df,
            x="min_vaf",
            y="Possible ctDNA+",
            color="min_variants",
            markers=True,
            title=title,
        )
        return fig
    except Exception as e:
        log.error("plot_threshold_sen_failed", error=str(e))
        return go.Figure()


def decision_curve_analysis(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    thresholds: np.ndarray | None = None,
) -> dict:
    """Compute Decision Curve Analysis (DCA) net benefit data.

    For each threshold, calculates the net benefit of using the model vs
    treating all or treating none. This helps clinicians choose an operating
    threshold that balances false positives against missed detections.

    Args:
        y_true: Binary ground truth labels (0/1).
        y_prob: Predicted probabilities for positive class.
        thresholds: Array of decision thresholds to evaluate.
            Defaults to ``np.linspace(0.01, 0.99, 99)``.

    Returns:
        Dictionary with keys ``thresholds``, ``net_benefit_model``, and
        ``net_benefit_treat_all``.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)

    n = len(y_true)
    prevalence = float(y_true.mean())
    net_benefits = []
    treat_all = []

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        tp = int(np.sum((y_pred == 1) & (y_true == 1)))
        fp = int(np.sum((y_pred == 1) & (y_true == 0)))
        odds = t / (1 - t) if t < 1 else float("inf")
        nb = (tp / n) - (fp / n) * odds
        net_benefits.append(float(nb))
        ta = prevalence - (1 - prevalence) * odds
        treat_all.append(float(ta))

    return {
        "thresholds": thresholds.tolist(),
        "net_benefit_model": net_benefits,
        "net_benefit_treat_all": treat_all,
    }

In [ ]:
# | export
def single_feature_model(
    X: np.ndarray,  # shape (n_samples, n_sub_metrics)
    y: np.ndarray,  # binary labels (1 = positive class)
    feature_names: list[str] | None = None,
    cancer_types: np.ndarray | None = None,
    assays: np.ndarray | None = None,
    n_folds: int = 5,
    random_state: int = 42,
) -> tuple[dict, object, object, object]:
    """Train LR, RF, and XGB on a feature matrix with stratified CV.

    Returns (results_dict, lr_pipeline, rf_model, xgb_model).

    Fixes applied (audit v3):
      - C-01: LR uses Pipeline(scaler+lr) to prevent data leakage
      - C-02: Subgroup metrics use out-of-fold predictions (unbiased)
      - H-01: LR has class_weight="balanced", XGB has scale_pos_weight
      - H-07: Bare except replaced with Exception
      - M-02: Bootstrap 95% CI on AUC values
    """
    import time
    from sklearn.metrics import classification_report, confusion_matrix

    try:
        from xgboost import XGBClassifier
    except Exception:
        # XGBoost may fail with XGBoostError if libomp is missing (macOS)
        XGBClassifier = None

    results = {}
    try:
        y = y.astype(int)
        if len(np.unique(y)) < 2:
            log.warning("single_class_y", y_unique=np.unique(y).tolist())
            return {"error": "Only one class present in target array"}, None, None, None

        # Guard for small groups where n_splits doesn't work
        min_class_counts = np.bincount(y).min()
        folds = min(n_folds, min_class_counts)
        if folds < 2:
            return (
                {"error": "Not enough samples per class for Stratified CV (<2)"},
                None,
                None,
                None,
            )

        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=random_state)
        results["cv_folds_actual"] = folds

        # --- Helper: Youden's J optimal threshold ---
        def get_optimal_threshold(y_true, y_pred_prob):
            """Find threshold maximizing Youden's J (sensitivity + specificity - 1)."""
            try:
                fpr, tpr, thresholds = roc_curve(y_true, y_pred_prob)
                if len(thresholds) > 0:
                    optimal_idx = np.argmax(tpr - fpr)
                    return float(thresholds[optimal_idx])
            except Exception:
                # roc_curve can fail with degenerate inputs (all same class)
                log.debug("optimal_threshold_fallback", reason="roc_curve failed")
            return 0.5

        def safely_get_metrics(y_true, y_pred_prob, threshold=0.5):
            """Compute classification report and confusion matrix at given threshold."""
            y_pred = (y_pred_prob >= threshold).astype(int)
            rep = classification_report(
                y_true, y_pred, output_dict=True, zero_division=0
            )
            cm = confusion_matrix(y_true, y_pred).tolist()
            return rep, cm

        def _bootstrap_auc(y_true, y_score, n_boot=1000, seed=42):
            """Compute 95% bootstrap CI for AUC-ROC."""
            rng = np.random.RandomState(seed)
            aucs = []
            for _ in range(n_boot):
                idx = rng.randint(0, len(y_true), len(y_true))
                if len(np.unique(y_true[idx])) < 2:
                    continue
                aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
            if len(aucs) < 10:
                return None, None
            return float(np.percentile(aucs, 2.5)), float(np.percentile(aucs, 97.5))

        # ── Logistic Regression (Pipeline prevents scaler leakage) ──
        lr_pipe = Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "lr",
                    LogisticRegression(
                        max_iter=1000,
                        random_state=random_state,
                        class_weight="balanced",
                    ),
                ),
            ]
        )

        lr_probs = cross_val_predict(lr_pipe, X, y, cv=cv, method="predict_proba")[:, 1]
        results["auc_lr"] = roc_auc_score(y, lr_probs)
        lr_ci_low, lr_ci_high = _bootstrap_auc(y, lr_probs)
        results["auc_lr_ci_lower"] = lr_ci_low
        results["auc_lr_ci_upper"] = lr_ci_high
        lr_opt = get_optimal_threshold(y, lr_probs)
        results["lr_optimal_threshold"] = lr_opt
        lr_rep, lr_cm = safely_get_metrics(y, lr_probs, threshold=lr_opt)
        results["lr_classification_report"] = lr_rep
        results["lr_confusion_matrix"] = lr_cm

        t0 = time.perf_counter()
        lr_pipe.fit(X, y)
        results["lr_training_time_sec"] = time.perf_counter() - t0
        results["lr_coef_direction"] = (
            "positive" if lr_pipe.named_steps["lr"].coef_[0].mean() > 0 else "negative"
        )

        # ── Random Forest ──
        rf = RandomForestClassifier(
            n_estimators=100,
            max_depth=5,
            min_samples_leaf=max(1, min(10, len(y) // 10)),
            random_state=random_state,
            class_weight="balanced",
        )
        rf_probs = cross_val_predict(rf, X, y, cv=cv, method="predict_proba")[:, 1]
        results["auc_rf"] = roc_auc_score(y, rf_probs)
        rf_ci_low, rf_ci_high = _bootstrap_auc(y, rf_probs)
        results["auc_rf_ci_lower"] = rf_ci_low
        results["auc_rf_ci_upper"] = rf_ci_high

        try:
            from sklearn.calibration import calibration_curve

            prob_true, prob_pred = calibration_curve(
                y, rf_probs, n_bins=10, strategy="uniform"
            )
            results["rf_calibration"] = {
                "prob_true": prob_true.tolist(),
                "prob_pred": prob_pred.tolist(),
            }
        except Exception:
            pass
        rf_opt = get_optimal_threshold(y, rf_probs)
        results["rf_optimal_threshold"] = rf_opt
        rf_rep, rf_cm = safely_get_metrics(y, rf_probs, threshold=rf_opt)
        results["rf_classification_report"] = rf_rep
        results["rf_confusion_matrix"] = rf_cm

        t0 = time.perf_counter()
        rf.fit(X, y)
        results["rf_training_time_sec"] = time.perf_counter() - t0
        if feature_names is not None:
            results["rf_feature_importances"] = dict(
                zip(feature_names, rf.feature_importances_.tolist())
            )
        else:
            results["rf_feature_importances"] = rf.feature_importances_.tolist()

        # ── XGBoost ──
        xgb = None
        xgb_probs = None
        if XGBClassifier is not None:
            pos_weight = len(y[y == 0]) / max(1, len(y[y == 1]))
            xgb = XGBClassifier(
                n_estimators=100,
                max_depth=5,
                learning_rate=0.1,
                random_state=random_state,
                eval_metric="logloss",
                use_label_encoder=False,
                scale_pos_weight=pos_weight,
            )
            try:
                xgb_probs = cross_val_predict(xgb, X, y, cv=cv, method="predict_proba")[
                    :, 1
                ]
                results["auc_xgb"] = roc_auc_score(y, xgb_probs)
                xgb_ci_low, xgb_ci_high = _bootstrap_auc(y, xgb_probs)
                results["auc_xgb_ci_lower"] = xgb_ci_low
                results["auc_xgb_ci_upper"] = xgb_ci_high
                xgb_opt = get_optimal_threshold(y, xgb_probs)
                results["xgb_optimal_threshold"] = xgb_opt
                xgb_rep, xgb_cm = safely_get_metrics(y, xgb_probs, threshold=xgb_opt)
                results["xgb_classification_report"] = xgb_rep
                results["xgb_confusion_matrix"] = xgb_cm

                t0 = time.perf_counter()
                xgb.fit(X, y)
                results["xgb_training_time_sec"] = time.perf_counter() - t0
            except Exception as e:
                log.warning("xgboost_cv_failed", error=str(e))
                xgb = None
                xgb_probs = None

        # ── AUC deltas ──
        results["auc_delta_rf_lr"] = results["auc_rf"] - results["auc_lr"]
        if "auc_xgb" in results:
            results["auc_delta_xgb_rf"] = results["auc_xgb"] - results["auc_rf"]

        # ── Top features (post-CV, from RF importances) ──
        if feature_names is not None and isinstance(
            results.get("rf_feature_importances"), dict
        ):
            sorted_feats = sorted(
                results["rf_feature_importances"].items(),
                key=lambda x: x[1],
                reverse=True,
            )
            results["top_features"] = [f[0] for f in sorted_feats[:10]]

        # ── PR curves (precision-recall) ──
        try:
            from sklearn.metrics import precision_recall_curve, average_precision_score

            for prefix, probs in [
                ("lr", lr_probs),
                ("rf", rf_probs),
                ("xgb", xgb_probs),
            ]:
                if probs is not None:
                    prec, rec, _ = precision_recall_curve(y, probs)
                    results[f"{prefix}_pr_curve"] = {
                        "precision": prec.tolist(),
                        "recall": rec.tolist(),
                    }
                    results[f"{prefix}_avg_precision"] = float(
                        average_precision_score(y, probs)
                    )
        except Exception as e:
            log.warning("pr_curve_failed", error=str(e))

        # ── Fold-level AUC tracking ──
        try:
            from sklearn.model_selection import cross_val_score

            rf_fold_aucs = cross_val_score(rf, X, y, cv=cv, scoring="roc_auc")
            results["rf_fold_aucs"] = rf_fold_aucs.tolist()
            results["rf_auc_std"] = float(rf_fold_aucs.std())
            log.debug(
                "fold_aucs_computed",
                rf_mean=f"{rf_fold_aucs.mean():.3f}",
                rf_std=f"{rf_fold_aucs.std():.3f}",
            )
        except Exception as e:
            log.warning("fold_auc_tracking_failed", error=str(e))

        # ── Feature stability across CV folds ──
        if feature_names is not None:
            try:
                from collections import Counter
                from sklearn.base import clone

                fold_top10 = Counter()
                for train_idx, _ in cv.split(X, y):
                    fold_rf = clone(rf).fit(X[train_idx], y[train_idx])
                    top10_idx = np.argsort(fold_rf.feature_importances_)[-10:]
                    for idx in top10_idx:
                        fold_top10[feature_names[idx]] += 1
                results["feature_stability"] = {
                    k: round(v / folds, 2) for k, v in fold_top10.items()
                }
            except Exception as e:
                log.warning("feature_stability_failed", error=str(e))

        # ── Threshold sensitivity sweep (RF) ──
        try:
            thresholds_sweep = np.linspace(0.01, 0.99, 50)
            sweep_data = []
            for t in thresholds_sweep:
                y_pred = (rf_probs >= t).astype(int)
                tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
                sweep_data.append(
                    {
                        "threshold": round(float(t), 3),
                        "sensitivity": float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0,
                        "specificity": float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0,
                        "ppv": float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0,
                    }
                )
            results["rf_threshold_sweep"] = sweep_data
        except Exception as e:
            log.warning("threshold_sweep_failed", error=str(e))

        # ── Decision Curve Analysis ──
        try:
            for prefix, probs in [("rf", rf_probs), ("xgb", xgb_probs)]:
                if probs is not None:
                    results[f"{prefix}_dca"] = decision_curve_analysis(y, probs)
        except Exception as e:
            log.warning("dca_failed", error=str(e))

        # ── Subgroup Analysis using OOF predictions (C-02: unbiased) ──
        # Use out-of-fold CV predictions, NOT the retrained model
        rf_oof_preds = (rf_probs >= rf_opt).astype(int)
        xgb_oof_preds = (
            (xgb_probs >= results.get("xgb_optimal_threshold", 0.5)).astype(int)
            if xgb_probs is not None
            else None
        )

        def _subgroup_metrics(mask, y_all, rf_preds, xgb_preds_arr):
            """Compute sensitivity for RF and XGB on a subgroup using OOF predictions."""
            res = {}
            y_sub = y_all[mask]
            if len(y_sub) < 5 or len(np.unique(y_sub)) < 2:
                return res

            # RF subgroup
            try:
                p_sub = rf_preds[mask]
                tn, fp, fn, tp = confusion_matrix(y_sub, p_sub, labels=[0, 1]).ravel()
                res["rf_sensitivity"] = float(tp / (tp + fn)) if (tp + fn) > 0 else None
                res["rf_specificity"] = float(tn / (tn + fp)) if (tn + fp) > 0 else None
            except Exception as e:
                log.warning("subgroup_rf_failed", error=str(e))

            # XGB subgroup
            if xgb_preds_arr is not None:
                try:
                    p_sub = xgb_preds_arr[mask]
                    tn, fp, fn, tp = confusion_matrix(
                        y_sub, p_sub, labels=[0, 1]
                    ).ravel()
                    res["xgb_sensitivity"] = (
                        float(tp / (tp + fn)) if (tp + fn) > 0 else None
                    )
                    res["xgb_specificity"] = (
                        float(tn / (tn + fp)) if (tn + fp) > 0 else None
                    )
                except Exception as e:
                    log.warning("subgroup_xgb_failed", error=str(e))

            return res

        # Cancer type subgroup analysis (Top 10)
        if cancer_types is not None:
            import pandas as pd

            c_stats = []
            c_series = pd.Series(cancer_types)
            top_10 = c_series.value_counts().nlargest(10).index

            for c_type in top_10:
                mask = cancer_types == c_type
                n_samp = mask.sum()
                if n_samp < 5:
                    continue
                res_sub = _subgroup_metrics(mask, y, rf_oof_preds, xgb_oof_preds)
                if res_sub:
                    c_stats.append(
                        {
                            "cancer_type": c_type,
                            "n_samples": int(n_samp),
                            "n_positives": int(y[mask].sum()),
                            **res_sub,
                        }
                    )
            results["cancer_type_stats"] = c_stats

        # Assay subgroup analysis
        if assays is not None:
            import pandas as pd

            a_stats = []
            a_series = pd.Series(assays)
            for a_type in a_series.dropna().unique():
                mask = assays == a_type
                n_samp = mask.sum()
                if n_samp < 5:
                    continue
                res_sub = _subgroup_metrics(mask, y, rf_oof_preds, xgb_oof_preds)
                if res_sub:
                    a_stats.append(
                        {
                            "assay": a_type,
                            "n_samples": int(n_samp),
                            "n_positives": int(y[mask].sum()),
                            **res_sub,
                        }
                    )
            results["assay_stats"] = a_stats

        return results, lr_pipe, rf, xgb
    except Exception as e:
        import traceback

        log.error(
            "single_feature_model_failed", error=str(e), trace=traceback.format_exc()
        )
        return {"error": str(e)}, None, None, None